# Cascadia Scoring — YOLO Model Training

Trains a YOLO detector on the synthetic dataset generated by `tools/dataset_genetor.py`.

Prerequisites:
1. Generate the dataset first, e.g. `python tools/dataset_genetor.py -train 100 -val 20`.
2. The dataset lives in `datasets/for_model/{train,val}/{images,labels}`.

Run the cells from top to bottom.

In [ ]:
!pip install -q ultralytics

## 1. Select architecture

Edit `MODEL_NAME` in the next cell to pick a different YOLO architecture.

| Model | Sizes |
|---|---|
| YOLO11 | `yolo11n` `yolo11s` `yolo11m` `yolo11l` `yolo11x` |
| YOLOv8 | `yolov8n` `yolov8s` `yolov8m` `yolov8l` `yolov8x` |

`n` is the fastest and smallest; `x` is the biggest and most accurate. Pretrained COCO weights are downloaded automatically on first use.

In [ ]:
# Select a YOLO architecture:
#   YOLO11 : yolo11n, yolo11s, yolo11m, yolo11l, yolo11x
#   YOLOv8 : yolov8n, yolov8s, yolov8m, yolov8l, yolov8x
MODEL_NAME = "yolo11n"

## 2. Dataset config

Reads the `config.yml` generated by the dataset tool, fixes the dataset `path`, remaps the class ids from 1–5 to 0–4 (YOLO requires 0-indexed classes), and writes a `data.yaml` for ultralytics.

In [ ]:
import yaml
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
DATASET_DIR = (NOTEBOOK_DIR / ".." / "datasets" / "for_model").resolve()

train_img_dir = DATASET_DIR / "train" / "images"
train_lbl_dir = DATASET_DIR / "train" / "labels"
val_img_dir = DATASET_DIR / "val" / "images"
val_lbl_dir = DATASET_DIR / "val" / "labels"

missing = [p for p in (train_img_dir, train_lbl_dir, val_img_dir, val_lbl_dir) if not p.exists()]
assert not missing, f"Dataset incomplete. Run tools/dataset_genetor.py first. Missing: {missing}"

# Load the config generated by tools/dataset_genetor.py.
with open(DATASET_DIR / "config.yml") as f:
    data = yaml.safe_load(f)

# Ultralytics requires 0-indexed classes; remap label ids 1..5 -> 0..4 once.
names = data["names"]  # dict keyed 1..5 as generated
sorted_ids = sorted(names.keys())
if sorted_ids != list(range(len(sorted_ids))):
    class_map = {old: new for new, old in enumerate(sorted_ids)}
    for lbl_dir in (train_lbl_dir, val_lbl_dir):
        for txt in lbl_dir.glob("*.txt"):
            lines = []
            for line in txt.read_text().splitlines():
                parts = line.split()
                parts[0] = str(class_map[int(parts[0])])
                lines.append(" ".join(parts))
            txt.write_text("\n".join(lines) + "\n")
    names = {class_map[k]: v for k, v in names.items()}

# Write a YOLO-compatible data.yaml.
data_yaml = DATASET_DIR / "data.yaml"
data["path"] = str(DATASET_DIR)
data["train"] = "train"
data["val"] = "val"
data["names"] = dict(sorted(names.items()))
data["nc"] = len(names)
with open(data_yaml, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print(f"data.yaml written to {data_yaml}")
print(f"classes: {data['names']}")

for split, img_dir, lbl_dir in [("train", train_img_dir, train_lbl_dir), ("val", val_img_dir, val_lbl_dir)]:
    n_img = len(list(img_dir.glob("*.png")))
    n_lbl = len(list(lbl_dir.glob("*.txt")))
    n_obj = sum(1 for txt in lbl_dir.glob("*.txt") for _ in txt.read_text().splitlines())
    print(f"{split}: {n_img} images, {n_lbl} label files, {n_obj} objects")

## 3. Hyperparameters

Tweak these in the next cell. With a small dataset, use a small `BATCH` and consider fewer `EPOCHS`.

In [ ]:
import torch

EPOCHS = 100
IMGSZ = 640
BATCH = 8
WORKERS = 2
DEVICE = 0 if torch.cuda.is_available() else "cpu"
PROJECT = "runs"
NAME = "cascadia"

print(f"Training on device: {DEVICE}")

## 4. Train

Downloads the pretrained checkpoint on first run, then fine-tunes on the generated dataset. Results land in `runs/cascadia/`.

In [ ]:
from ultralytics import YOLO

model = YOLO(f"{MODEL_NAME}.pt")

model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    device=DEVICE,
    project=PROJECT,
    name=NAME,
)

## 5. Evaluate

Runs the trained model on the validation split and prints mAP / precision / recall.

In [ ]:
metrics = model.val()

print(f"mAP50:   {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

## 6. Save & test

Copies the best weights into `output/` and shows a quick prediction on a validation image.

In [ ]:
import shutil

best_path = model.trainer.best
output_dir = (NOTEBOOK_DIR / ".." / "output").resolve()
output_dir.mkdir(parents=True, exist_ok=True)
out_model = output_dir / f"cascadia_{MODEL_NAME}.pt"
shutil.copy(best_path, out_model)
print(f"Best weights saved to {out_model}")

In [ ]:
import matplotlib.pyplot as plt

results = model.predict(source=val_img_dir, imgsz=IMGSZ, conf=0.25, verbose=False)

for r in results[:1]:
    im = r.plot()
    plt.figure(figsize=(10, 10))
    plt.imshow(im[..., ::-1])
    plt.axis("off")
    plt.show()